# Perhitungan TF-IDF dan Word Embeding (Data Berita)

In [ ]:
%%capture
!pip install plotly
!pip install --upgrade gensim

In [ ]:
!pip install --upgrade --force-reinstall numpy gensim


  Using cached numpy-2.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached gensim-4.3.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (8.1 kB)
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached scipy-1.13.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached smart_open-7.3.1-py3-none-any.whl.metadata (24 kB)
  Using cached wrapt-1.17.3-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (6.4 kB)
Using cached gensim-4.3.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (26.6 MB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
Using cached scipy-1.13.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (38.2 MB)
Using cached smart_open-7.3.1-py3-none-any.whl (61 kB)
Using cached wrapt-1.17.3-cp312-cp312-manylinux1_x86_6

In [ ]:
from gensim.models import Word2Vec, FastText
import pandas as pd
import re

from sklearn.decomposition import PCA

from matplotlib import pyplot as plt
import plotly.graph_objects as go

import numpy as np

import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('berita_detik_preprocessed.csv')

In [ ]:
from gensim.models import Word2Vec

In [ ]:
import numpy as np

class MyTokenizer:
    def fit_transform(self, texts):
        # Tokenisasi sederhana: lowercase + split
        return [str(text).lower().split() for text in texts]

class MeanEmbeddingVectorizer:
    def __init__(self, word2vec_model):
        self.word2vec = word2vec_model
        # Perbaikan: gunakan vector_size (Gensim ≥ 4.0)
        self.dim = word2vec_model.wv.vector_size

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_tokenized = MyTokenizer().fit_transform(X)
        embeddings = []
        for words in X_tokenized:
            # Ambil vektor hanya untuk kata yang ada di vocab
            valid_vectors = [
                self.word2vec.wv[word] for word in words
                if word in self.word2vec.wv
            ]
            if valid_vectors:
                embeddings.append(np.mean(valid_vectors, axis=0))
            else:
                embeddings.append(np.zeros(self.dim))
        return np.array(embeddings)

    def fit_transform(self, X, y=None):
        return self.transform(X)

In [ ]:
clean_txt = []
for w in range(len(df.isi)):
    # ubah ke string dulu, isi NaN jadi string kosong
    desc = str(df['isi'][w]).lower()

    # remove punctuation
    desc = re.sub('[^a-zA-Z]', ' ', desc)

    # remove tags
    desc = re.sub("&lt;/?.*?&gt;", " ", desc)

    # remove digits and special chars
    desc = re.sub("(\\d|\\W)+", " ", desc)

    clean_txt.append(desc)

df['tokens_final'] = clean_txt
df.head()


,id,judul,link,kategori,isi,before,after_lower_no_symbol,after_stopword,after_corrected,after_stemmed,tokens_final,array,embedding_length,sentence_embedding
0,1,"Dua Tahun Berlalu, Korban Gempa Maroko Masih H...",https://news.detik.com/foto-news/d-8114261/dua...,news,"Maroko - Dua tahun pascagempa, ribuan warga Ma...",maroko dua tahun pascagempa ribuan warga marok...,maroko dua tahun pascagempa ribuan warga marok...,maroko pascagempa ribuan warga maroko tinggal ...,maroko pascagempa ribuan harga maroko tinggal ...,maroko pascagempa ribu harga maroko tinggal te...,maroko dua tahun pascagempa ribuan warga marok...,"[[-0.010699586, 0.017314877, -0.0060127103, -0...",22,"[-6.4685584e-05, 0.00032119677, 0.0002946688, ..."
1,2,Alvi Pemutilasi Pacar Diamuk dan Diumpat Warga...,https://news.detik.com/berita/d-8116286/alvi-p...,news,Rekonstruksi kasus Alvi Maulana (24) yang muti...,rekonstruksi kasus alvi maulana yang mutilasi ...,rekonstruksi kasus alvi maulana yang mutilasi ...,rekonstruksi alvi maulana mutilasi tiara angel...,rekonstruksi alvi maulana mutilasi tiara angel...,rekonstruksi alvi maulana mutilasi tiara angel...,rekonstruksi kasus alvi maulana yang mutilasi ...,"[[-0.013450207, -0.018679561, 0.0009880124, -0...",88,"[0.0031934495, -0.0019993037, 0.0015487056, 0...."
2,3,"Truk Seruduk 2 Angkot Lagi Ngetem di Bogor, 3 ...",https://news.detik.com/berita/d-8116283/truk-s...,news,Kecelakaan lalu lintas melibatkan truk dan dua...,kecelakaan lalu lintas melibatkan truk dan dua...,kecelakaan lalu lintas melibatkan truk dan dua...,kecelakaan lintas melibatkan truk unit angkot ...,kecelakaan lintas melibatkan truk tni angkot a...,celaka lintas libat truk tni angkot angkut kot...,kecelakaan lalu lintas melibatkan truk dan dua...,"[[-0.0027899707, -0.017694484, 0.0027040553, -...",82,"[-0.00019099689, -0.0028137479, 0.00026481214,..."
3,4,MK Gelar Sidang Putusan 5 Gugatan UU TNI Hari Ini,https://news.detik.com/berita/d-8116272/mk-gel...,news,Mahkamah Konstitusi (MK) akan menggelar sidang...,mahkamah konstitusi mk akan menggelar sidang g...,mahkamah konstitusi mk akan menggelar sidang g...,mahkamah konstitusi mk menggelar sidang gugata...,mahkamah konstitusi kpk menggelar sidang gugat...,mahkamah konstitusi kpk gelar sidang gugat uu ...,mahkamah konstitusi mk akan menggelar sidang g...,"[[-0.0022926165, 0.0037466213, -0.009113402, 0...",74,"[0.0025689802, 0.00036175258, 0.002266981, 0.0..."
4,5,"Gelar Razia, Pemprov Banten Temukan 86 Kendara...",https://news.detik.com/berita/d-8116269/gelar-...,news,Pemerintah Provinsi Banten menggelar razia ken...,pemerintah provinsi banten menggelar razia ken...,pemerintah provinsi banten menggelar razia ken...,pemerintah provinsi banten menggelar razia ken...,pemerintah provinsi banten menggelar razia ken...,perintah provinsi banten gelar razia kendara b...,pemerintah provinsi banten menggelar razia ken...,"[[0.00038313336, -0.011171429, 0.020692125, 0....",118,"[0.0019087285, 0.00020290892, 0.002978689, 0.0..."


In [ ]:
df.shape

(150, 14)

In [ ]:
corpus = []
for col in df['tokens_final']:
    word_list = col.split(" ")
    corpus.append(word_list)

# show first value
print(corpus[0:1])

# generate vectors from corpus
model = Word2Vec(corpus, min_count=1, vector_size=50)


[['maroko', 'dua', 'tahun', 'pascagempa', 'ribuan', 'warga', 'maroko', 'masih', 'tinggal', 'di', 'tenda', 'darurat', 'mereka', 'menuntut', 'bantuan', 'lebih', 'besar', 'di', 'tengah', 'gencarnya', 'proyek', 'stadion', '']]


In [ ]:
from gensim.models import Word2Vec

# cek dulu apakah kata ada di vocabulary
def safe_most_similar(word):
    if word in model.wv:
        return model.wv.most_similar(word)
    else:
        return f"'{word}' tidak ada di vocabulary"

print(safe_most_similar('eric'))

# gunakan cosmul dengan pengecekan
positive_words = ['phone', 'number']
negative_words = ['call']

if all(w in model.wv for w in positive_words + negative_words):
    print(model.wv.most_similar_cosmul(positive=positive_words, negative=negative_words))
else:
    print("Ada kata yang tidak ada di vocabulary untuk cosmul test")

# outlier detection (pengganti doesnt_match)
words = "phone number prison cell".split()
valid_words = [w for w in words if w in model.wv]

if len(valid_words) > 1:
    print("Outlier:", model.wv.get_outlier(valid_words))
else:
    print("Tidak cukup kata valid di vocabulary untuk outlier test")

# save embeddings
filename = 'berita_embd.txt'
model.wv.save_word2vec_format(filename, binary=False)
print(f"Embeddings berhasil disimpan di {filename}")


'eric' tidak ada di vocabulary
Ada kata yang tidak ada di vocabulary untuk cosmul test
Tidak cukup kata valid di vocabulary untuk outlier test
Embeddings berhasil disimpan di berita_embd.txt


In [ ]:
mean_embedding_vectorizer = MeanEmbeddingVectorizer(model)
mean_embedded = mean_embedding_vectorizer.fit_transform(df['before'])

In [ ]:
df['array'] = df['before'].apply(
    lambda x: [model.wv[word] for word in x.split() if word in model.wv]
)


In [ ]:
df['embedding_length'] = df['array'].apply(len)


In [ ]:
print(df.columns)   # untuk lihat kolom apa saja yang ada
df.head()


Index(['id', 'judul', 'link', 'kategori', 'isi', 'before',
       'after_lower_no_symbol', 'after_stopword', 'after_corrected',
       'after_stemmed', 'tokens_final', 'array', 'embedding_length',
       'sentence_embedding'],
      dtype='object')


,id,judul,link,kategori,isi,before,after_lower_no_symbol,after_stopword,after_corrected,after_stemmed,tokens_final,array,embedding_length,sentence_embedding
0,1,"Dua Tahun Berlalu, Korban Gempa Maroko Masih H...",https://news.detik.com/foto-news/d-8114261/dua...,news,"Maroko - Dua tahun pascagempa, ribuan warga Ma...",maroko dua tahun pascagempa ribuan warga marok...,maroko dua tahun pascagempa ribuan warga marok...,maroko pascagempa ribuan warga maroko tinggal ...,maroko pascagempa ribuan harga maroko tinggal ...,maroko pascagempa ribu harga maroko tinggal te...,maroko dua tahun pascagempa ribuan warga marok...,"[[-0.010696203, 0.017314568, -0.0060050604, -0...",22,"[-6.4685584e-05, 0.00032119677, 0.0002946688, ..."
1,2,Alvi Pemutilasi Pacar Diamuk dan Diumpat Warga...,https://news.detik.com/berita/d-8116286/alvi-p...,news,Rekonstruksi kasus Alvi Maulana (24) yang muti...,rekonstruksi kasus alvi maulana yang mutilasi ...,rekonstruksi kasus alvi maulana yang mutilasi ...,rekonstruksi alvi maulana mutilasi tiara angel...,rekonstruksi alvi maulana mutilasi tiara angel...,rekonstruksi alvi maulana mutilasi tiara angel...,rekonstruksi kasus alvi maulana yang mutilasi ...,"[[-0.013432026, -0.018676031, 0.0009931973, -0...",88,"[0.0031934495, -0.0019993037, 0.0015487056, 0...."
2,3,"Truk Seruduk 2 Angkot Lagi Ngetem di Bogor, 3 ...",https://news.detik.com/berita/d-8116283/truk-s...,news,Kecelakaan lalu lintas melibatkan truk dan dua...,kecelakaan lalu lintas melibatkan truk dan dua...,kecelakaan lalu lintas melibatkan truk dan dua...,kecelakaan lintas melibatkan truk unit angkot ...,kecelakaan lintas melibatkan truk tni angkot a...,celaka lintas libat truk tni angkot angkut kot...,kecelakaan lalu lintas melibatkan truk dan dua...,"[[-0.0027693189, -0.017717171, 0.002769074, -0...",82,"[-0.00019099689, -0.0028137479, 0.00026481214,..."
3,4,MK Gelar Sidang Putusan 5 Gugatan UU TNI Hari Ini,https://news.detik.com/berita/d-8116272/mk-gel...,news,Mahkamah Konstitusi (MK) akan menggelar sidang...,mahkamah konstitusi mk akan menggelar sidang g...,mahkamah konstitusi mk akan menggelar sidang g...,mahkamah konstitusi mk menggelar sidang gugata...,mahkamah konstitusi kpk menggelar sidang gugat...,mahkamah konstitusi kpk gelar sidang gugat uu ...,mahkamah konstitusi mk akan menggelar sidang g...,"[[-0.0022939423, 0.0037371365, -0.009071284, 0...",74,"[0.0025689802, 0.00036175258, 0.002266981, 0.0..."
4,5,"Gelar Razia, Pemprov Banten Temukan 86 Kendara...",https://news.detik.com/berita/d-8116269/gelar-...,news,Pemerintah Provinsi Banten menggelar razia ken...,pemerintah provinsi banten menggelar razia ken...,pemerintah provinsi banten menggelar razia ken...,pemerintah provinsi banten menggelar razia ken...,pemerintah provinsi banten menggelar razia ken...,perintah provinsi banten gelar razia kendara b...,pemerintah provinsi banten menggelar razia ken...,"[[0.00042605802, -0.01120574, 0.02077001, 0.01...",118,"[0.0019087285, 0.00020290892, 0.002978689, 0.0..."


In [ ]:
df.shape

(150, 14)

In [ ]:
import numpy as np

# buat kalimat embedding dengan mean dari semua kata
df['sentence_embedding'] = df['array'].apply(
    lambda x: np.mean(x, axis=0) if len(x) > 0 else np.zeros(model.vector_size)
)

# konversi ke DataFrame
embedding_df = pd.DataFrame(df['sentence_embedding'].tolist())

print(embedding_df.head())
print("Shape:", embedding_df.shape)  # (jumlah_kalimat, vector_size)


         0         1         2         3         4         5         6   \
0 -0.000016  0.000306  0.000417  0.013333 -0.009994 -0.017485  0.033354   
1  0.003241 -0.002006  0.001657  0.011149 -0.008941 -0.015097  0.027763   
2 -0.000144 -0.002825  0.000375  0.010797 -0.009627 -0.018936  0.025676   
3  0.002632  0.000347  0.002414  0.014739 -0.018027 -0.024658  0.038524   
4  0.001958  0.000192  0.003095  0.013206 -0.013844 -0.017455  0.029572   

         7         8         9   ...        40        41        42        43  \
0  0.026911 -0.028236 -0.002173  ...  0.030321 -0.011065 -0.007183  0.006229   
1  0.026313 -0.025098 -0.006337  ...  0.025824 -0.004995 -0.001400  0.004931   
2  0.028540 -0.027923 -0.006741  ...  0.027873 -0.009754 -0.002848  0.004348   
3  0.039078 -0.035122 -0.008852  ...  0.031914 -0.009393 -0.003663  0.002950   
4  0.026797 -0.029366 -0.007183  ...  0.027627 -0.007629 -0.000602  0.002915   

         44        45        46        47        48        49  
0  0

In [ ]:
print(df.columns)


Index(['id', 'judul', 'link', 'kategori', 'isi', 'before',
       'after_lower_no_symbol', 'after_stopword', 'after_corrected',
       'after_stemmed', 'tokens_final', 'array', 'embedding_length',
       'sentence_embedding'],
      dtype='object')


In [ ]:
embedding_df.shape

(150, 50)